# Validacion mano de obra: Informe de Costo vs ODS

Cruza el **Informe de Costo** (nomina) contra el registro de la **ODS**
(empleados del contrato), por numero de documento. Por cada campo del `MAPEO`
(texto, fecha o numero):

- si los dos valores **coinciden** -> una lista de **un solo elemento** `[valor]`,
- si **difieren** -> una lista con **ambos valores** `[valor_informe, valor_ods]`.

La columna **`Estado revision`** es `"ok"` cuando todos coinciden, o
`"valores no coinciden: <campos>"`.

Notas: el encabezado real del Informe esta en la fila 10; varias columnas con
tildes llegan corruptas en el xlsx, por eso se referencian **por posicion**; la
orden de servicio se extrae del texto de `Nombre Centro Costo` (`...Os050...` -> `50`).
Esta es la version de exploracion; la logica de produccion vive en `mano_obra.py`.


In [1]:
import re
import unicodedata

import pandas as pd


In [2]:
# --- Lectura del Informe (renombrando columnas usadas por posicion) ---
path_informe = "docs/manodeobra/1.Informe de Costo CTABARCA junio 2025.xlsx"
path_ods = "docs/manodeobra/050/Empleados contrato N° 3023604 (junio - 2025).xlsx"

INFORME_COLS_POR_POSICION = {
    0: "Tipo de pago",
    2: "Identificacion",
    4: "Nombres",
    5: "Apellidos",
    6: "Cargo",
    10: "Nombre Centro Costo",
    14: "Fecha de Ingreso",
    15: "Fecha de retiro",
    16: "Dias Trabajados",
}

informe_df = pd.read_excel(path_informe, sheet_name="Informe", header=9)
informe_df = informe_df.rename(
    columns={informe_df.columns[i]: nombre for i, nombre in INFORME_COLS_POR_POSICION.items()}
)
ods_df = pd.read_excel(path_ods)

print("Informe:", informe_df.shape, "| ODS:", ods_df.shape)


Informe: (106, 58) | ODS: (211, 72)


In [3]:
# --- Normalizacion y helpers de comparacion ---
def solo_digitos(valor):
    """Deja solo los digitos (91.499.442 -> 91499442)."""
    return re.sub(r"\D", "", str(valor))


def norm_texto(valor):
    """Mayusculas, sin acentos y espacios colapsados, para comparar texto."""
    if pd.isna(valor):
        return ""
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", texto)


def extraer_os(valor):
    """Orden de servicio desde 'Nombre Centro Costo': '...Os050...' -> 50."""
    if pd.isna(valor):
        return None
    m = re.search(r"OS\s*0*(\d+)", str(valor), re.IGNORECASE)
    return int(m.group(1)) if m else None


def _es_vacio(valor):
    if valor is None:
        return True
    try:
        return bool(pd.isna(valor))
    except (TypeError, ValueError):
        return False


def presentacion(valor, tipo):
    """Valor que se muestra en la celda (fecha ISO, entero limpio o texto)."""
    if tipo == "fecha":
        fecha = pd.to_datetime(valor, errors="coerce")
        return "" if pd.isna(fecha) else fecha.strftime("%Y-%m-%d")
    if tipo == "numero":
        if _es_vacio(valor):
            return ""
        try:
            numero = float(str(valor).replace(",", "").strip())
            return str(int(numero)) if numero.is_integer() else str(numero)
        except ValueError:
            return re.sub(r"\s+", "", str(valor))
    return "" if _es_vacio(valor) else str(valor)


def comparable(valor, tipo):
    """Forma normalizada que decide si dos valores coinciden."""
    if tipo == "texto":
        return norm_texto(valor)
    return presentacion(valor, tipo)


In [4]:
# --- Mapeo de columnas equivalentes ---
# (etiqueta, columna Informe, columna ODS, tipo: 'texto' | 'fecha' | 'numero')
MAPEO = [
    ("OS", "OS", "No_de_orden_de_servicio_conocido_por_el_contratista", "numero"),
    ("Nombres", "Nombres", "Nombres", "texto"),
    ("Apellidos", "Apellidos", "Apellidos", "texto"),
    ("Cargo", "Cargo", "CargoContratoLaboral", "texto"),
    ("Fecha de Ingreso", "Fecha de Ingreso",
     "Fecha_de_inicio_de_actividades_del_trabajador_para_el_contrato_comercial_u_orden_de_servicio", "fecha"),
    ("Fecha de retiro", "Fecha de retiro",
     "Fecha_fin_de_actividades_del_trabajador_para_el_contrato_comercial_u_orden_de_servicio", "fecha"),
    ("D\u00edas Trabajados", "Dias Trabajados", "DiasTrabajadosEnMes", "numero"),
]

COL_DOC_INFORME = "Identificacion"
COL_DOC_ODS = "NumeroDocumento"


In [5]:
# --- Construccion del DataFrame de validacion ---
inf = informe_df.copy()
ods = ods_df.copy()

# Del Informe solo interesan las filas de Recobro (no las de Nomina).
inf = inf[inf["Tipo de pago"].map(norm_texto) == "RECOBRO"]

inf["OS"] = inf["Nombre Centro Costo"].map(extraer_os)
inf["_doc"] = inf[COL_DOC_INFORME].map(solo_digitos)
ods["_doc"] = ods[COL_DOC_ODS].map(solo_digitos)

# Quitar filas sin documento y duplicados (cada lado puede repetir a la persona).
inf = inf[inf["_doc"] != ""].drop_duplicates("_doc")
ods = ods[ods["_doc"] != ""].drop_duplicates("_doc")
ods_por_doc = ods.set_index("_doc")

filas = []
for _, registro in inf.iterrows():
    doc = registro["_doc"]
    if doc not in ods_por_doc.index:
        continue  # persona del Informe que no esta en la ODS
    otro = ods_por_doc.loc[doc]

    fila = {"Documento": doc}
    campos_diferentes = []
    for etiqueta, col_inf, col_ods, tipo in MAPEO:
        val_inf = registro.get(col_inf)
        val_ods = otro.get(col_ods)
        disp_inf = presentacion(val_inf, tipo)
        disp_ods = presentacion(val_ods, tipo)
        if comparable(val_inf, tipo) == comparable(val_ods, tipo):
            fila[etiqueta] = [disp_inf]            # coinciden -> un solo elemento
        else:
            fila[etiqueta] = [disp_inf, disp_ods]  # difieren -> ambos valores
            campos_diferentes.append(etiqueta)

    if campos_diferentes:
        fila["Estado revisión"] = "valores no coinciden: " + ", ".join(campos_diferentes)
    else:
        fila["Estado revisión"] = "ok"
    filas.append(fila)

columnas = ["Documento"] + [m[0] for m in MAPEO] + ["Estado revisión"]
resultado = pd.DataFrame(filas, columns=columnas)

total = len(resultado)
ok = (resultado["Estado revisión"] == "ok").sum() if total else 0
print(f"Personas cruzadas: {total}  |  OK: {ok}  |  Con diferencias: {total - ok}")


Personas cruzadas: 10  |  OK: 0  |  Con diferencias: 10


In [6]:
# --- Resultado ---
pd.set_option("display.max_colwidth", 60)
resultado


,Documento,OS,Nombres,Apellidos,Cargo,Fecha de Ingreso,Fecha de retiro,Días Trabajados,Estado revisión
0,1096238423,[50],[Jesus Manuel],[Ruiz Escaño],[Almacenista],"[2025-06-01, 2024-10-19]","[2025-06-30, 2025-08-15]","[30, 29]","valores no coinciden: Fecha de Ingreso, Fecha de retiro,..."
1,13888018,"[56, 50]",[Alvaro Enrique],[Castillo Caceres],[Almacenista],"[2025-06-01, 2025-06-08]","[2025-06-30, 2025-08-15]","[30, 21]","valores no coinciden: OS, Fecha de Ingreso, Fecha de ret..."
2,1140819437,[50],[Nelson Eduardo],[Fernandez Ortiz],[Auxiliar De Materiales],"[2025-06-01, 2024-03-26]","[2025-06-30, 2025-08-15]","[30, 28]","valores no coinciden: Fecha de Ingreso, Fecha de retiro,..."
3,1096194776,[50],[Orosman],[Rodriguez Villamizar],[Operador Camion Grua],"[2025-06-01, 2025-03-04]","[2025-06-30, 2025-08-15]","[30, 29]","valores no coinciden: Fecha de Ingreso, Fecha de retiro,..."
4,13541671,[50],[Franz Josef],[Arias Corena],"[Operador Camion Grua, Operador de camion grua]",[2025-06-14],"[2025-06-30, 2025-08-15]","[17, 16]","valores no coinciden: Cargo, Fecha de retiro, Días Traba..."
5,1096193775,[50],[Jhon Anderson],[Madarriaga Rivera],[Rescatista],"[2025-06-01, 2025-03-04]","[2025-06-30, 2025-08-15]","[30, 28]","valores no coinciden: Fecha de Ingreso, Fecha de retiro,..."
6,1096229847,"[54, 50]",[Brayan Stiven],[Perez Villarreal],[Rescatista],"[2025-06-01, 2025-06-08]","[2025-06-30, 2025-08-15]","[30, 21]","valores no coinciden: OS, Fecha de Ingreso, Fecha de ret..."
7,13569198,[50],[Jose Yaisinio],[Peña Guzman],[Rescatista],"[2025-06-01, 2022-02-22]","[2025-06-30, 2025-08-15]","[30, 29]","valores no coinciden: Fecha de Ingreso, Fecha de retiro,..."
8,91499442,"[54, 50]",[Miguel Angel],[Becerra Lopez],[Rescatista],"[2025-06-01, 2025-06-08]","[2025-06-30, 2025-08-15]","[30, 23]","valores no coinciden: OS, Fecha de Ingreso, Fecha de ret..."
9,93478958,[50],[Victor Manuel],[Matoma Lozano],[Rescatista],"[2025-06-01, 2022-09-12]","[2025-06-30, 2025-08-15]","[30, 26]","valores no coinciden: Fecha de Ingreso, Fecha de retiro,..."
